# Matemáticas de la Inteligencia Artificial
## Sesión 7 — Optimización, generalización y el experimento completo de aprendizaje

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/07_pytorch_generalizacion/laboratorio.ipynb)

### Pregunta de la sesión
**¿Por qué minimizar el error de entrenamiento no garantiza haber aprendido bien y cómo se convierte nuestro entrenamiento manual en un experimento moderno con PyTorch?**

Este cuaderno está generado como JSON válido de Jupyter y evita secuencias de escape problemáticas en las celdas Markdown.

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
print('PyTorch:', torch.__version__)

# 1. Gradiente manual frente a autograd

Calcula primero con NumPy el gradiente de un clasificador lineal multiclase y comprueba después que `loss.backward()` reproduce el mismo resultado.

Para la cross-entropy con softmax, el gradiente respecto de los logits es `(P-Y)/N`.

In [ ]:
X_np = np.array([[1.,.5],[-.5,1.5],[.7,-1.],[1.2,.1],[-1.,-.7]], dtype=np.float32)
y_np = np.array([0,1,2,0,1], dtype=np.int64)
W_np = np.array([[.20,-.10,.05],[-.30,.25,.10]], dtype=np.float32)
b_np = np.array([.02,-.03,.01], dtype=np.float32)

Z = X_np @ W_np + b_np
# TODO: softmax estable por filas
P = ...
Y = np.zeros_like(P)
Y[np.arange(len(y_np)), y_np] = 1.0
# TODO: pérdida y gradientes manuales
loss_np = ...
dZ = ...
gW_np = ...
gb_np = ...

X_t = torch.tensor(X_np)
y_t = torch.tensor(y_np)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
# TODO: forward
logits_t = ...
loss_t = nn.functional.cross_entropy(logits_t, y_t)
# TODO: backward
...

print('loss NumPy =', loss_np)
print('loss PyTorch =', loss_t.item())
print('error max dW =', np.max(np.abs(gW_np-W_t.grad.detach().numpy())))
print('error max db =', np.max(np.abs(gb_np-b_t.grad.detach().numpy())))

# 2. Train, validation y test

Usaremos un problema sintético de tres clases. La normalización se calcula exclusivamente con train para evitar data leakage.

In [ ]:
def make_spiral(n_per_class=220, noise=.22, seed=7):
    rng = np.random.default_rng(seed)
    Xs, ys = [], []
    for k in range(3):
        r = np.linspace(.08, 1., n_per_class)
        th = 4.2*r + 2*np.pi*k/3 + rng.normal(0., noise, size=n_per_class)
        Xs.append(np.c_[r*np.cos(th), r*np.sin(th)])
        ys.append(np.full(n_per_class, k, dtype=np.int64))
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    p = rng.permutation(len(y))
    return X[p], y[p]

def split_indices(N, train_frac=.60, val_frac=.20, seed=7):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(N)
    ntr = int(train_frac*N)
    nva = int(val_frac*N)
    # TODO: devuelve índices de train, validation y test
    return ..., ..., ...

X, y = make_spiral()
itr, iva, ite = split_indices(len(X), seed=SEED)
Xtr, ytr = X[itr], y[itr]
Xva, yva = X[iva], y[iva]
Xte, yte = X[ite], y[ite]

# TODO: media y desviación SOLO de train
mu = ...
sd = ...
Xtr = ...
Xva = ...
Xte = ...
print(len(Xtr), len(Xva), len(Xte))

# 3. Tensores, mini-batches y MLP

Construye una red de dos capas ocultas. La salida final debe contener tres logits.

In [ ]:
def ds(X, y):
    return TensorDataset(torch.tensor(X,dtype=torch.float32), torch.tensor(y,dtype=torch.long))

train_loader = DataLoader(ds(Xtr,ytr), batch_size=32, shuffle=True)
val_loader = DataLoader(ds(Xva,yva), batch_size=128, shuffle=False)
test_loader = DataLoader(ds(Xte,yte), batch_size=128, shuffle=False)

class MLP(nn.Module):
    def __init__(self, hidden=(32,32), dropout=0.0):
        super().__init__()
        layers = []
        din = 2
        for dout in hidden:
            # TODO: Linear + Tanh
            layers += [...]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            din = dout
        # TODO: capa final con tres logits
        layers.append(...)
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return ...

model = MLP().to(device)
print(model)

# 4. Ciclo moderno de entrenamiento

Orden conceptual: `zero_grad -> forward -> loss -> backward -> step`.

Recuerda: `backward()` calcula gradientes y `step()` actualiza parámetros.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_ok = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        # TODO: zero_grad, forward, loss, backward, step
        ...
        n = len(yb)
        total_loss += loss.item()*n
        total_ok += (logits.argmax(1)==yb).sum().item()
        total_n += n
    return total_loss/total_n, total_ok/total_n

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_ok = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        # TODO: forward y loss, sin backward ni step
        ...
        n = len(yb)
        total_loss += loss.item()*n
        total_ok += (logits.argmax(1)==yb).sum().item()
        total_n += n
    return total_loss/total_n, total_ok/total_n

# 5. Sobreajuste, regularización y early stopping

Entrena una red deliberadamente grande con pocos datos. Después compara con una versión regularizada mediante AdamW, weight decay, dropout y early stopping.

El criterio de selección debe usar validation, nunca test.

In [ ]:
# TODO: construye aquí el experimento de sobreajuste y regularización.
# Guarda las curvas train_loss y val_loss para representarlas.

# Problema final abierto — Diseñar un experimento de aprendizaje fiable

Genera un nuevo dataset con `make_spiral(n_per_class=180, noise=0.28, seed=2026)` y construye al menos dos modelos candidatos.

## Entregables
1. Arquitectura y optimizador de cada candidato.
2. Curvas de train y validation.
3. Diagnóstico de underfitting u overfitting.
4. Selección del modelo exclusivamente con validation.
5. Evaluación de test una única vez después de cerrar la elección.
6. Generalization gap: test loss menos train loss.
7. Matriz de confusión 3x3 sin sklearn.
8. Frontera de decisión final.
9. Dos posibles formas de data leakage.
10. Conclusión: qué evidencia tienes de que el modelo generaliza.

> El problema es abierto: no existe una única arquitectura correcta.

In [ ]:
# TU TRABAJO EMPIEZA AQUÍ
# No uses test hasta haber seleccionado definitivamente el modelo con validation.